# Opintolainan takaisinmaksusuunnitelmien vertailu PROC LOAN -proseduurilla

## Yhteenveto

Opintotukitoimisto arvioi, miten valmistuva vuosikurssi kannattaisi maksaa takaisin edustava 27 500 dollarin saldo vertailemalla neljää takaisinmaksurakennetta — liittovaltion kiinteäkorkoista vakiolainaa, yksityistä jälleenrahoitusta aloitusmaksulla, korkokattoista vaihtuvakorkoista (ARM) lainaa sekä työnantajan tukemaa korkoalennuslainaa — käyttäen `PROC LOAN` -proseduuria. 120 kuukauden laina-ajalla nämä neljä tarjousta asettuvat selkeään järjestykseen kuukausierän ja kokonaiskoron osalta niiden ilmoitetuilla alkukoroilla: liittovaltion vakiolaina maksaa eniten korkoa (**10 021,22 dollaria** korolla 6,53 %, erä **312,68 dollaria**), yksityinen jälleenrahoitus laskee korkoa mutta lisää 412,50 dollarin maksun (**9 120,20 dollaria** korolla 5,99 %, erä **305,17 dollaria**), ja alemman koron ARM-laina (**7 099,76 dollaria** korolla 4,75 %, erä **288,33 dollaria**) sekä työnantajan korkoalennuslaina (**6 700,67 dollaria** korolla 4,50 %, erä **285,01 dollaria**) tuottavat pienimmät korkokulut. `COMPARE`-lohko raportoi sitten jokaisen suunnitelman kertyneen koron, lyhennetyn pääoman ja jäljellä olevan saldon 36, 60 ja 120 kuukauden kohdalla, mikä näyttää, kuinka pitkälle kukin laina on lyhentynyt niissä kohdissa, joissa lainanottaja todennäköisimmin jälleenrahoittaa tai maksaa lainan pois.

## Tietolähteet

| Aineisto | Rivit | Kuvaus | Keskeiset muuttujat |
|---------|------|-------------|---------------|
| `borrowers` | 40 | Synteettiset valmistuvan vuosikurssin lainaprofiilit, jotka luodaan suoraan koodissa komennoilla `call streaminit(20260531)` ja `rand('uniform')`. Käytetään realististen lainaehtojen määrittämiseen vertailua varten. | `student_id` (1001-1040), `balance` (pääoma valmistumishetkellä; tämä otos vaihtelee välillä 11 800-47 750 dollaria, keskiarvo 30 771 dollaria), `apr` (nimelliskorko 4,10-9,10 % vuodessa, keskiarvo 6,50 %), `term` (120 tai 180 kuukautta, keskiarvo 144), `origination_pct` (aloitusmaksu 1,0-2,0 %, keskiarvo 1,50 %) |

Edustava lainanottaja, jota analysoidaan `PROC LOAN`-proseduurilla (summa 27 500 dollaria, 120 kuukauden laina-aika, aloitus heinäkuussa 2026), sijoittuu tämän ryhmän saldo- ja korkojakauman alempaan keskivaiheeseen; ulkoista dataa tai verkkodataa ei käytetä. Vuosikurssi on olemassa uskottavien lainaehtojen motivoimiseksi — varsinainen suora vertailu tehdään yhdelle edustavalle lainalle.

# Opintolainan takaisinmaksusuunnitelmien vertailu PROC LOAN -proseduurilla

Kun opiskelijat valmistuvat, opintotukitoimiston on autettava heitä valitsemaan kilpailevien takaisinmaksutarjousten joukosta. `PROC LOAN` (SAS/ETS) on suunniteltu juuri tähän: se lyhentää kiinteäkorkoisia, vaihtuvakorkoisia (ARM) ja korkoalennuslainoja ja vertailee niitä sitten rinnakkain kuukausierän, kokonaiskoron ja lyhennysasteen suhteen.

Tässä muistikirjassa:

1. Luodaan synteettinen valmistuva vuosikurssi realististen lainaehtojen määrittämiseksi.
2. Tehdään yhteenveto ryhmästä `PROC MEANS`-proseduurilla.
3. Rakennetaan neljä takaisinmaksusuunnitelmaa edustavalle 27 500 dollarin saldolle ja vertaillaan niitä `PROC LOAN`-proseduurilla (FIXED / ARM / BUYDOWN + COMPARE).
4. Ajetaan suositeltu kiinteäkorkoinen suunnitelma vielä erikseen sen itsenäisen taloudellisuuden vahvistamiseksi.

Kaikki suoritetaan paikallisesti sisäänrakennetulla synteettisellä datalla — ei ulkoisia tiedostoja tai verkkoyhteyttä.

## Vaihe 1 — Synteettisen valmistuvan vuosikurssin luominen

Simuloimme 40 lainanottajaa. Jokaiselle arvotaan pääomasaldo valmistumishetkellä, luottokelpoisuuteen löyhästi sidottu nimelliskorko, vakiomuotoinen takaisinmaksuaika (10 tai 15 vuotta) sekä aloitusmaksuprosentti. Siemenluku tekee datasta toistettavaa.

In [1]:
TIEDOT borrowers;
   CALL streaminit(20260531);
   PITUUS plan $ 28;
   TEE student_id = 1001 ASTI 1040;
      /* Pääomasaldo valmistumishetkellä: 9500-48000 dollaria */
      balance = round(9500 + 38500*rand('uniform'), 50);
      /* Vuotuinen nimelliskorko luottoluokan mukaan: 4,0-9,5 % */
      apr = round(4.0 + 5.5*rand('uniform'), 0.05);
      /* Vakiomuotoinen takaisinmaksuaika: 120 tai 180 kuukautta */
      JOS rand('uniform') < 0.6 NIIN term = 120;
      MUUTEN term = 180;
      /* Aloitusmaksu prosentteina pääomasta: 1,0-2,0 % */
      origination_pct = round(1.0 + rand('uniform'), 0.10);
      TULOSTE;
   LOPPU;
SUORITA;


NOTE: DATA borrowers


NOTE: Wrote borrowers (40 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Vaihe 2 — Ryhmän profilointi

Ennen yksittäisten lainojen mallintamista tarkastelemme saldojen, korkojen ja laina-aikojen jakaumaa. Tämä kertoo, millainen *edustava* lainanottaja on — se on perusta seuraavalle suoralle vertailulle.

In [2]:
PROSEDUURI KESKIARVOT TIEDOT=borrowers n mean MIN MAX maxdec=2;
   MUUTTUJA balance apr term origination_pct;
   NIMIKE balance='Saldo' apr='Korko (%)' term='Laina-aika (kk)' origination_pct='Aloitusmaksu (%)';
SUORITA;

                                                  The MEANS Procedure

 Variable         Label                   N           Mean     Minimum     Maximum
 ---------------------------------------------------------------------------------
 balance          Saldo                  40       30771.25    11800.00    47750.00
 apr              Korko (%)              40           6.50        4.10        9.10
 term             Laina-aika (kk)        40         144.00      120.00      180.00
 origination_pct  Aloitusmaksu (%)       40           1.50        1.00        2.00
 ---------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Vaihe 3 — Neljän takaisinmaksusuunnitelman vertailu edustavalle lainanottajalle

Käyttäen edustavaa 27 500 dollarin saldoa 120 kuukauden (10 vuoden) laina-ajalla, joka alkaa heinäkuussa 2026, asetamme rinnakkain neljä realistista tarjousta:

- **Liittovaltion suora tukematon laina (vakio)** — tavallinen kiinteäkorkoinen laina korolla 6,53 %.
- **Yksityinen jälleenrahoitus (maksullinen)** — alempi 5,99 %:n kiinteä korko, mutta 412,50 dollarin aloituskustannus (`INIT=`).
- **Yksityinen vaihtuvakorkoinen ARM-laina** — 4,75 %:n vaihtuvakorkoinen laina, jossa on 1 %:n muutoskohtainen / 5 %:n elinkaarikatto `CAPS=`, 9,75 %:n `MAXRATE=`, vuosittainen `ADJUSTFREQ=` ja avainsana `WORSTCASE`.
- **Työnantajan 2-1-korkoalennus** — tuettu 4,50 %:n alkukorko, joka ilmoitetun aikataulun mukaan nousee `BDRATES=`-määrityksellä 5,50 %:iin toisena vuonna ja 6,50 %:iin sen jälkeen.

`COMPARE`-lauseke pyytää lainojenvälistä vertailunäkymää 36, 60 ja 120 kuukauden kohdalla, 22 %:n `TAXRATE=`-verokannalla ja 4 %:n vähimmäistuottovaatimuksella (`MARR=`); `OUTSUM=` ja `OUTCOMP=` tallentavat lainakohtaisen yhteenvedon ja vertailurivit. Alla oleva tuloste raportoi jokaiselle suunnitelmalle ja tarkistuspisteelle **kertyneen maksetun koron, kertyneen lyhennetyn pääoman ja jäljellä olevan saldon** — selkeän näkymän lyhennysasteesta ehdokkaiden välillä.

> **Huomio korkomuutoksista.** Jennerin `PROC LOAN` lyhentää jokaisen suunnitelman sen ilmoitetulla **alkukorolla**, joten ARM-lainan `CAPS`/`MAXRATE`/`WORSTCASE`-asetukset ja korkoalennuslainan `BDRATES`-portaat näkyvät tulosteessa lainan ehtoina, mutta niitä **ei** sovelleta maksulaskentaan — alla olevat ARM- ja korkoalennuslukemat kuvastavat niiden 4,75 %:n ja 4,50 %:n alkukorkoja pidettynä muuttumattomina koko laina-ajan. Käsittele näitä kahta kokonaissummaa parhaan tapauksen (alkukoron) kustannuksina, ei pahimman tapauksen kattoina.

In [3]:
PROSEDUURI loan START=2026:7 amount=27500 life=120 outsum=plan_sum;

   fixed rate=6.53
         label='Liittovaltion suora tukematon laina (vakio)';

   fixed rate=5.99 init=412.50
         label='Yksityinen jälleenrahoitus (maksullinen)';

   arm rate=4.75 caps=(1 5) maxrate=9.75 adjustfreq=12 worstcase
       label='Yksityinen vaihtuvakorkoinen ARM-laina (pahin tapaus)';

   buydown rate=4.50 bdrates=(13=5.50 25=6.50)
           label='Työnantajan 2-1-korkoalennus, sitten 6,5 %';

   COMPARE at=(36 60 120) ALL taxrate=22 marr=4 OUTCOMP=plan_cmp;
SUORITA;

                            The LOAN Procedure

  Number of loans evaluated:    4

  Loan #1: Liittovaltion suora tukematon laina (vakio)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan #2: Yksityinen jälleenrahoitus (maksullinen)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            5.9900
    Life (months):                          120
    Monthly Payment:                     305.17
    Total Paid (P&I):                  36620.20
    Total Interest:                     9120.20
    Initialization Cost:                 412.50

  Loan #3: Yksityinen vaihtuvakorkoinen ARM-laina (pahin tapaus)
    Loan Type:                   Adjustab


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Vaihe 4 — Suositellun kiinteäkorkoisen suunnitelman ajaminen erikseen

Lainanottajalle, joka arvostaa maksun varmuutta, liittovaltion kiinteäkorkoinen vakiosuunnitelma on turvallinen oletusvalinta. Ajamme sen erikseen vahvistaaksemme sen keskeiset tunnusluvut: sama **312,68 dollarin** kuukausierä, **37 521,22 dollarin** kokonaissumma ja **10 021,22 dollarin** kokonaiskorko, jotka nähtiin neljän suunnitelman vertailussa, esitettynä nyt itsenäisenä lainayhteenvetona.

In [4]:
PROSEDUURI loan START=2026:7 amount=27500 rate=6.53 life=120 schedule=yearly;
   fixed label='Liittovaltion suora tukematon laina (vakio)';
SUORITA;

                            The LOAN Procedure

  Number of loans evaluated:    1

  Loan #1: Liittovaltion suora tukematon laina (vakio)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan Repayment Schedule: Liittovaltion suora tukematon laina (vakio)
      Year    Begin Balance        Payment       Interest      Principal      End Balance
    ------ ---------------- -------------- -------------- -------------- ----------------
         1         27500.00        3752.12        1736.12        2016.00         25484.00
         2         25484.00        3752.12        1600.47        2151.66         23332.34
         3         23332.34        3752.12        1455.68        2296.44         21035.90
         4   


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Tulosten tulkinta

Kaikki neljä suunnitelmaa lyhentävät saman 27 500 dollarin pääoman kokonaan 120 kuukaudessa (jokaisen suunnitelman jäljellä oleva saldo on 0,00 dollaria kuukaudella 120 COMPARE-taulukossa), joten päätös riippuu **kuukausierästä** ja **kokonaiskorosta ilmoitetulla korolla**:

| Suunnitelma | Korko | Erä | Kokonaiskorko | Huomiot |
|------|------|---------|----------------|-------|
| Liittovaltion suora vakio | 6,53 % | 312,68 $ | 10 021,22 $ | Ei maksua; vahvin lainanottajan suoja |
| Yksityinen jälleenrahoitus | 5,99 % | 305,17 $ | 9 120,20 $ | 412,50 $ aloitusmaksu |
| Yksityinen vaihtuvakorkoinen ARM | 4,75 % (alku) | 288,33 $ | 7 099,76 $ | Korko voi nousta; luku on alkukoron mukainen kustannus |
| Työnantajan 2-1-korkoalennus | 4,50 % (alku) | 285,01 $ | 6 700,67 $ | Riippuu työsuhteen jatkumisesta |

- **Liittovaltion vakiosuunnitelma** on kallein koron osalta (10 021,22 $), mutta tarjoaa kiinteän, ennustettavan 312,68 dollarin erän ilman maksua.
- **Yksityinen jälleenrahoitus** laskee korkoa 5,99 %:iin (9 120,20 $ korkoa, 901 dollaria vähemmän kuin liittovaltion suunnitelmassa), mutta veloittaa 412,50 dollarin aloitusmaksun, joten sen nettoetu liittovaltion suunnitelmaan nähden on noin 488 dollaria korkoa vähennettynä maksulla — merkityksellistä vain, jos laina kestää tarpeeksi kauan ilman jälleenrahoitusta.
- **ARM-laina** ja **korkoalennuslaina** näyttävät tässä pienimmän koron (7 099,76 $ ja 6 700,67 $) yksinomaan siksi, että niiden **alkukorot** ovat selvästi kiinteitä tarjouksia alempia. Kuten vaiheessa 3 todettiin, Jenner pitää nämä alkukorot muuttumattomina, joten nämä ovat parhaan tapauksen lukuja: todellinen ARM-laina, joka nousee, tai korkoalennuslaina, joka menettää työnantajan tuen, päätyisi korkeammalle, eikä erää ole taattu.

`COMPARE`-taulukko tarkentaa tätä näyttämällä, kuinka nopeasti kukin suunnitelma lyhenee. 36 kuukauden kohdalla liittovaltion suunnitelma on maksanut 4 792,27 dollaria korkoa ja lyhentänyt 6 464,10 dollaria pääomaa (saldo 21 035,90 $), kun taas korkoalennuslaina on maksanut vain 3 263,97 dollaria korkoa ja lyhentänyt 6 996,24 dollaria pääomaa (saldo 20 503,76 $) — matalamman koron suunnitelmat sekä maksavat vähemmän korkoa että lyhentävät pääomaa nopeammin ensimmäisten kolmen vuoden aikana. Riskiä karttavalle vastavalmistuneelle liittovaltion vakiosuunnitelman korkovarmuus oikeuttaa usein sen korkeamman koron; lainanottajalle, joka luottaa vakaaseen ja pitkäaikaiseen työsuhteeseen, korkoalennuslainan tuettu alku tuo suurimmat säästöt niiden vaihtoehtojen joukossa, jotka todella lukitsevat korkonsa.